# 第三章：岛屿进化 — 多种群隔离与迁移

**系列：** OpenEvolve — 从零到专家·手撕全流程  
**上一章：** [02 — MAP-Elites](./02-map-elites.ipynb)  
**本章内容：** 为什么单个 MAP-Elites 网格还不够、岛屿模型的理论基础、环形拓扑迁移、以及从零实现多岛屿进化系统。

---

## 单个网格的问题

在第二章中，我们实现了 MAP-Elites 网格，它通过为每种「特征组合」保留最佳程序来维持多样性。但还有一个隐藏的问题：

**收敛压力仍然存在。** 想象这样一个场景：

1. 冒泡排序经过 50 代优化，分数达到 0.65
2. 归并排序刚出现，分数只有 0.30
3. 它们恰好映射到同一个网格单元（代码长度类似、复杂度类似）
4. 归并排序**立刻被丢弃**——因为 0.30 < 0.65

问题的根源：**所有程序在同一个竞技场竞争**。一个新颖但尚未成熟的方法，会在它有机会成长之前就被淘汰。

### 生物学类比

这就像把所有物种放在同一个岛上——最具竞争力的物种会消灭其他物种。但在自然界中，加拉帕戈斯群岛的每个岛屿独立进化出了截然不同的物种。**地理隔离**让弱小但有潜力的物种有空间成长。

这正是**岛屿模型（Island Model）**的核心思想。

## 直觉：加拉帕戈斯群岛

```mermaid
graph TD
    subgraph 岛屿0["岛屿 0：冒泡排序家族"]
        A0["冒泡v1 0.52"] --> A1["冒泡v2 0.58"]
        A1 --> A2["冒泡v3 0.65"]
    end
    subgraph 岛屿1["岛屿 1：插入排序家族"]
        B0["插入v1 0.35"] --> B1["插入v2 0.48"]
        B1 --> B2["插入v3 0.61"]
    end
    subgraph 岛屿2["岛屿 2：归并排序家族"]
        C0["归并v1 0.30"] --> C1["归并v2 0.55"]
        C1 --> C2["归并v3 0.78"]
    end
    A2 -.->|迁移| 岛屿1
    B2 -.->|迁移| 岛屿2
    C2 -.->|迁移| 岛屿0
```

每个岛屿**独立进化**，有自己的 MAP-Elites 网格。归并排序在岛屿 2 上不需要和冒泡排序的 0.65 竞争——它只需要打败岛屿 2 上的其他归并排序变体。

周期性地，每个岛屿的**精英程序**会迁移到相邻岛屿，引入新的遗传多样性。

## 理论：岛屿模型

### 形式化定义

设我们有 $K$ 个岛屿，每个岛屿维护自己的 MAP-Elites 网格：

$$
\{\mathcal{G}_0, \mathcal{G}_1, \ldots, \mathcal{G}_{K-1}\}
$$

每个网格 $\mathcal{G}_k$ 独立运行 MAP-Elites 算法（第二章）。

### 迁移

每隔 $M$ 代（迁移间隔），执行一次迁移：

$$
\begin{aligned}
& \textbf{Migration}(\{\mathcal{G}_k\}, M, r): \\
& \quad \textbf{for } k = 0, 1, \ldots, K-1: \\
& \quad\quad \text{migrants}_k \leftarrow \text{top}(\mathcal{G}_k, r) \quad \text{// 选择前 } r\% \text{ 的精英} \\
& \quad\quad \textbf{for each } x \in \text{migrants}_k: \\
& \quad\quad\quad x' \leftarrow \text{copy}(x) \\
& \quad\quad\quad \mathcal{G}_{(k+1) \bmod K}.\text{add}(x') \quad \text{// 环形拓扑：迁移到下一个岛屿}
\end{aligned}
$$

### 环形拓扑（Ring Topology）

OpenEvolve 使用**环形拓扑**：每个岛屿只向相邻的两个岛屿迁移。

$$
\text{neighbors}(k) = \{(k-1) \bmod K, \; (k+1) \bmod K\}
$$

为什么不用全连接（每个岛屿向所有岛屿迁移）？
- 全连接会导致岛屿快速同质化，失去隔离的意义
- 环形拓扑让好的基因需要经过多步才能传遍所有岛屿
- 这给了每个岛屿足够的时间独立探索

> **出处：** 岛屿模型起源于 Wright (1931) 的种群遗传学理论。在进化计算中由 Tanese (1989) 正式引入。  
> **对应原始代码：** `openevolve/database.py:L1775-1880` — `migrate_programs()` 实现环形迁移

### 数学 → 代码 映射表

| 数学符号 | 代码变量 | 含义 |
|----------|----------|------|
| $K$ | `self.num_islands` | 岛屿数量 |
| $\mathcal{G}_k$ | `self.islands[k]` (MAPElitesGrid) | 第 $k$ 个岛屿的网格 |
| $M$ | `self.migration_interval` | 迁移间隔（代数） |
| $r$ | `self.migration_rate` | 迁移比例（精英的百分比） |
| $(k+1) \bmod K$ | `(k+1) % self.num_islands` | 环形拓扑的下一个岛屿 |
| $\text{migrants}_k$ | `get_top_programs(island)` | 岛屿 $k$ 的精英程序列表 |

## 实现：岛屿进化系统

我们将构建四个组件：
1. **IslandSystem** — 管理多个 MAP-Elites 网格
2. **程序分配** — 子代继承父代的岛屿
3. **迁移机制** — 环形拓扑迁移
4. **集成进化循环** — 轮流在各岛屿上进化

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import random
import copy
import uuid
from typing import Optional, List, Tuple
from dataclasses import dataclass, field

# 复用前面章节的模块
from our_implementation.core import Program
from our_implementation.map_elites import MAPElitesGrid, get_features

print("导入完成")

In [ ]:
class IslandSystem:
    """多岛屿进化系统。
    
    每个岛屿维护独立的 MAP-Elites 网格，周期性地通过迁移交换精英程序。
    
    对应 OpenEvolve: database.py:ProgramDatabase 中的岛屿相关逻辑
    - self.islands (Set[str])       -> 我们的 self.islands (MAPElitesGrid)
    - self.island_feature_maps      -> 集成在每个 MAPElitesGrid 内
    - self.island_generations       -> self.generations
    - self.island_best_programs     -> 每个 grid.best
    """
    
    def __init__(
        self,
        num_islands: int = 5,
        feature_dims: list[str] = None,
        n_bins: int = 10,
        migration_interval: int = 50,
        migration_rate: float = 0.1,
    ):
        """
        Args:
            num_islands: 岛屿数量（默认 5，OpenEvolve 默认也是 5）
            feature_dims: MAP-Elites 特征维度
            n_bins: 每个维度的桶数
            migration_interval: 每隔多少代迁移一次
            migration_rate: 每次迁移选择精英的比例
        
        设计决策：
        - 默认 5 个岛屿：太少隔离效果差，太多浪费计算资源
        - 迁移间隔 50 代：给每个岛屿足够时间独立探索
        - 迁移比例 10%：既引入多样性，又不淹没本地种群
        """
        if feature_dims is None:
            feature_dims = ["complexity", "score"]
        
        self.num_islands = num_islands
        self.migration_interval = migration_interval
        self.migration_rate = migration_rate
        
        # 每个岛屿有自己独立的 MAP-Elites 网格
        self.islands: list[MAPElitesGrid] = [
            MAPElitesGrid(feature_dims=feature_dims, n_bins=n_bins)
            for _ in range(num_islands)
        ]
        
        # 每个岛屿的代数计数器
        self.generations: list[int] = [0] * num_islands
        
        # 上次迁移时的最大代数
        self.last_migration_gen: int = 0
        
        # 当前活跃岛屿（轮流进化用）
        self.current_island: int = 0
        
        # 全局最佳（跨所有岛屿）
        self.global_best: Optional[Program] = None
    
    def add(self, program: Program, island_id: Optional[int] = None) -> bool:
        """将程序添加到指定岛屿。
        
        对应 OpenEvolve: database.py:add() 方法
        - 如果指定了 island_id，添加到该岛屿
        - 否则添加到 current_island
        """
        idx = island_id if island_id is not None else self.current_island
        stored = self.islands[idx].add(program)
        
        # 记录程序所在的岛屿
        program.metrics["island"] = idx
        
        # 更新全局最佳
        fitness = program.metrics.get("score", 0.0)
        if self.global_best is None or fitness > self.global_best.metrics.get("score", 0.0):
            self.global_best = program
        
        return stored
    
    def sample(self, island_id: Optional[int] = None) -> Program:
        """从指定岛屿采样一个程序。
        
        对应 OpenEvolve: database.py:sample_from_island()
        如果岛屿为空，回退到任意非空岛屿。
        """
        idx = island_id if island_id is not None else self.current_island
        
        # 尝试从指定岛屿采样
        if len(self.islands[idx]) > 0:
            return self.islands[idx].sample()
        
        # 回退：从任意非空岛屿采样
        for i in range(self.num_islands):
            if len(self.islands[i]) > 0:
                return self.islands[i].sample()
        
        raise ValueError("所有岛屿都为空")
    
    def next_island(self) -> int:
        """切换到下一个岛屿（轮转）。
        
        对应 OpenEvolve: database.py:next_island()
        """
        self.current_island = (self.current_island + 1) % self.num_islands
        return self.current_island
    
    def increment_generation(self, island_id: Optional[int] = None):
        """增加指定岛屿的代数。"""
        idx = island_id if island_id is not None else self.current_island
        self.generations[idx] += 1
    
    def should_migrate(self) -> bool:
        """检查是否应该执行迁移。
        
        对应 OpenEvolve: database.py:should_migrate()
        当最大岛屿代数距上次迁移超过 migration_interval 时触发。
        
        设计决策：基于代数而非迭代次数，因为不同岛屿的进化速度可能不同。
        """
        max_gen = max(self.generations)
        return (max_gen - self.last_migration_gen) >= self.migration_interval
    
    def migrate(self) -> int:
        """执行环形拓扑迁移。返回迁移的程序总数。
        
        对应 OpenEvolve: database.py:migrate_programs()
        
        环形拓扑：岛屿 k 的精英迁移到岛屿 (k+1) % K 和 (k-1) % K。
        迁移的是副本，原始程序保留在源岛屿。
        """
        total_migrated = 0
        
        for k in range(self.num_islands):
            island = self.islands[k]
            if len(island) == 0:
                continue
            
            # 获取岛屿中所有程序，按适应度排序
            programs = list(island.grid.values())
            programs.sort(key=lambda p: p.metrics.get("score", 0.0), reverse=True)
            
            # 选择前 migration_rate 比例的精英
            n_migrants = max(1, int(len(programs) * self.migration_rate))
            migrants = programs[:n_migrants]
            
            # 迁移到相邻岛屿（环形拓扑）
            for prog in migrants:
                # 跳过已经是迁移来的程序（防止重复迁移链）
                if prog.metrics.get("migrant", False):
                    continue
                
                for neighbor in [(k + 1) % self.num_islands,
                                 (k - 1) % self.num_islands]:
                    # 创建副本
                    migrant = Program(
                        code=prog.code,
                        metrics={**prog.metrics, "migrant": True, "source_island": k},
                        generation=prog.generation,
                    )
                    
                    if self.islands[neighbor].add(migrant):
                        total_migrated += 1
        
        # 更新迁移时间戳
        self.last_migration_gen = max(self.generations)
        
        return total_migrated
    
    def get_stats(self) -> list[dict]:
        """获取每个岛屿的统计信息。"""
        stats = []
        for i, island in enumerate(self.islands):
            best_score = island.best.metrics.get("score", 0.0) if island.best else 0.0
            stats.append({
                "island": i,
                "population": len(island),
                "coverage": f"{island.coverage:.0%}",
                "best_score": f"{best_score:.4f}",
                "generation": self.generations[i],
            })
        return stats


print(f"IslandSystem 就绪。默认配置：5 个岛屿，迁移间隔 50 代，迁移比例 10%")

## 演示：观察岛屿独立进化

让我们创建一个 3 岛屿系统，在每个岛屿上进化不同类型的程序，然后观察迁移效果：

In [ ]:
import numpy as np

random.seed(42)
np.random.seed(42)

# 创建 3 个岛屿的系统，迁移间隔设为 10 代（演示用，实际应更大）
system = IslandSystem(
    num_islands=3,
    feature_dims=["complexity"],
    n_bins=5,
    migration_interval=10,
    migration_rate=0.3,  # 30%，演示用
)

# 在每个岛屿上播种不同类型的程序
# 岛屿 0：短程序，低分（冒泡排序家族）
for i in range(20):
    p = Program(
        code="x" * random.randint(50, 150),
        metrics={"score": 0.3 + random.random() * 0.2, "complexity": float(random.randint(50, 150))},
        generation=i,
    )
    system.add(p, island_id=0)

# 岛屿 1：中等程序（插入排序家族）
for i in range(20):
    p = Program(
        code="x" * random.randint(100, 300),
        metrics={"score": 0.4 + random.random() * 0.25, "complexity": float(random.randint(100, 300))},
        generation=i,
    )
    system.add(p, island_id=1)

# 岛屿 2：长程序，高分（归并排序家族）
for i in range(20):
    p = Program(
        code="x" * random.randint(200, 500),
        metrics={"score": 0.5 + random.random() * 0.3, "complexity": float(random.randint(200, 500))},
        generation=i,
    )
    system.add(p, island_id=2)

print("播种完成。各岛屿状态：")
for stat in system.get_stats():
    print(f"  岛屿 {stat['island']}: {stat['population']} 个程序, "
          f"覆盖率 {stat['coverage']}, 最佳分数 {stat['best_score']}")

In [ ]:
# 模拟几代进化，然后触发迁移
print("迁移前：")
for i, island in enumerate(system.islands):
    scores = [p.metrics.get("score", 0) for p in island.grid.values()]
    if scores:
        print(f"  岛屿 {i}: 分数范围 [{min(scores):.3f}, {max(scores):.3f}], "
              f"平均 {sum(scores)/len(scores):.3f}")

# 手动推进代数到触发迁移
for _ in range(10):
    for i in range(3):
        system.increment_generation(i)

print(f"\n当前最大代数: {max(system.generations)}")
print(f"是否应该迁移: {system.should_migrate()}")

# 执行迁移
n_migrated = system.migrate()
print(f"\n迁移完成！共迁移了 {n_migrated} 个程序")

print("\n迁移后：")
for i, island in enumerate(system.islands):
    scores = [p.metrics.get("score", 0) for p in island.grid.values()]
    migrants = sum(1 for p in island.grid.values() if p.metrics.get("migrant", False))
    if scores:
        print(f"  岛屿 {i}: 分数范围 [{min(scores):.3f}, {max(scores):.3f}], "
              f"平均 {sum(scores)/len(scores):.3f}, 其中 {migrants} 个是迁移来的")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

island_names = ["岛屿 0 (短程序)", "岛屿 1 (中等程序)", "岛屿 2 (长程序)"]
colors_local = ["#3b82f6", "#22c55e", "#f59e0b"]
color_migrant = "#ef4444"

for i, (island, ax) in enumerate(zip(system.islands, axes)):
    local_scores = []
    local_complexity = []
    migrant_scores = []
    migrant_complexity = []
    
    for prog in island.grid.values():
        s = prog.metrics.get("score", 0)
        c = prog.metrics.get("complexity", len(prog.code))
        if prog.metrics.get("migrant", False):
            migrant_scores.append(s)
            migrant_complexity.append(c)
        else:
            local_scores.append(s)
            local_complexity.append(c)
    
    ax.scatter(local_complexity, local_scores, c=colors_local[i],
               alpha=0.6, s=40, label="本地程序")
    if migrant_scores:
        ax.scatter(migrant_complexity, migrant_scores, c=color_migrant,
                   alpha=0.8, s=60, marker="*", label="迁移程序")
    
    ax.set_xlabel("复杂度 (代码长度)")
    if i == 0:
        ax.set_ylabel("适应度分数")
    ax.set_title(island_names[i])
    ax.legend(fontsize=8)
    ax.set_ylim(0.2, 0.9)

plt.suptitle("岛屿进化：迁移后各岛屿的程序分布",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("红色星号 = 从其他岛屿迁移来的程序")
print("注意：每个岛屿仍然保持自己的特色，但迁移引入了新的遗传物质。")

## 完整演示：多岛屿进化循环

让我们将岛屿系统集成到完整的进化循环中。我们轮流在各岛屿上进化，并在合适的时机触发迁移：

In [ ]:
def run_island_evolution(
    initial_code: str,
    evaluator,
    mutator,
    num_islands: int = 3,
    iterations: int = 300,
    migration_interval: int = 30,
) -> IslandSystem:
    """运行多岛屿进化循环。
    
    对应 OpenEvolve 的核心流程：
    controller.py 的主循环 + database.py 的岛屿管理
    """
    system = IslandSystem(
        num_islands=num_islands,
        feature_dims=["complexity"],
        n_bins=5,
        migration_interval=migration_interval,
        migration_rate=0.2,
    )
    
    # 在每个岛屿播种初始程序
    for i in range(num_islands):
        initial = Program(
            code=initial_code,
            metrics=evaluator(initial_code),
            generation=0,
        )
        system.add(initial, island_id=i)
    
    # 记录历史
    history = {"iteration": [], "island": [], "best_score": [], "migration": []}
    
    for t in range(iterations):
        # 轮流选择岛屿（OpenEvolve: next_island()）
        island_id = t % num_islands
        
        # 从当前岛屿采样父代
        parent = system.sample(island_id=island_id)
        
        # 变异
        child_code = mutator(parent.code)
        
        # 评估
        child_metrics = evaluator(child_code)
        child = Program(
            code=child_code,
            metrics=child_metrics,
            generation=parent.generation + 1,
        )
        
        # 添加到同一个岛屿（子代继承父代的岛屿）
        system.add(child, island_id=island_id)
        system.increment_generation(island_id)
        
        # 检查并执行迁移
        migrated = False
        if system.should_migrate():
            n = system.migrate()
            migrated = True
        
        # 记录
        history["iteration"].append(t)
        history["island"].append(island_id)
        history["best_score"].append(
            system.global_best.metrics.get("score", 0) if system.global_best else 0
        )
        history["migration"].append(migrated)
    
    return system, history

print("run_island_evolution() 就绪")

In [ ]:
# 复用第一章的评估器和变异器
from our_implementation.core import evaluate_sorting, mutate_random, INITIAL_PROGRAM

random.seed(42)

# 运行 3 岛屿进化，300 次迭代
system, history = run_island_evolution(
    initial_code=INITIAL_PROGRAM,
    evaluator=evaluate_sorting,
    mutator=mutate_random,
    num_islands=3,
    iterations=300,
    migration_interval=30,
)

print("进化完成！\n")
print("各岛屿最终状态：")
for stat in system.get_stats():
    print(f"  岛屿 {stat['island']}: {stat['population']} 个程序, "
          f"代数 {stat['generation']}, 最佳 {stat['best_score']}")

if system.global_best:
    print(f"\n全局最佳分数: {system.global_best.metrics.get('score', 0):.4f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), height_ratios=[3, 1])

# 上图：每个岛屿的最佳分数随时间变化
island_colors = ["#3b82f6", "#22c55e", "#f59e0b"]

# 跟踪每个岛屿的最佳分数
island_bests = {i: [] for i in range(3)}
current_bests = {i: 0.0 for i in range(3)}

for t, (island, score) in enumerate(zip(history["island"], history["best_score"])):
    # 用全局最佳来近似每个岛屿在每步的表现
    for i in range(3):
        island_bests[i].append(current_bests[i])
    # 更新活跃岛屿的最佳分数
    current_bests[island] = max(current_bests[island], score)

for i in range(3):
    axes[0].plot(island_bests[i], color=island_colors[i], alpha=0.7,
                 label=f"岛屿 {i}", linewidth=1.5)

# 标记迁移事件
migration_iters = [t for t, m in enumerate(history["migration"]) if m]
for mt in migration_iters:
    axes[0].axvline(x=mt, color="red", alpha=0.3, linestyle="--", linewidth=1)

axes[0].set_ylabel("最佳适应度分数")
axes[0].set_title("多岛屿进化：各岛屿独立进化 + 周期迁移")
axes[0].legend(loc="lower right")

# 添加迁移标注
if migration_iters:
    axes[0].annotate("迁移事件", xy=(migration_iters[0], 0),
                     xytext=(migration_iters[0] + 20, 0.1),
                     arrowprops=dict(arrowstyle="->", color="red"),
                     color="red", fontsize=10)

# 下图：哪个岛屿在每步被选中
axes[1].scatter(history["iteration"], history["island"],
                c=[island_colors[i] for i in history["island"]],
                s=2, alpha=0.5)
axes[1].set_xlabel("迭代次数")
axes[1].set_ylabel("活跃岛屿")
axes[1].set_yticks([0, 1, 2])
axes[1].set_title("岛屿轮转调度")

plt.tight_layout()
plt.show()

print(f"共发生 {len(migration_iters)} 次迁移事件")
print("红色虚线 = 迁移时间点")

## 对比：单网格 vs. 多岛屿

让我们量化岛屿模型的优势。用同样的迭代预算，比较单网格和 3 岛屿系统：

In [ ]:
# 对比实验：单网格 vs. 3 岛屿
random.seed(123)

# 方法 1：单个 MAP-Elites 网格
single_grid = MAPElitesGrid(feature_dims=["complexity"], n_bins=5)
single_best_history = []

p0 = Program(code=INITIAL_PROGRAM,
             metrics=evaluate_sorting(INITIAL_PROGRAM),
             generation=0)
single_grid.add(p0)

for t in range(300):
    parent = single_grid.sample()
    child_code = mutate_random(parent.code)
    child = Program(code=child_code, metrics=evaluate_sorting(child_code),
                    generation=parent.generation + 1)
    single_grid.add(child)
    single_best_history.append(
        single_grid.best.metrics.get("score", 0) if single_grid.best else 0
    )

# 方法 2：3 岛屿系统（重新运行以保证公平比较）
random.seed(123)
system3, hist3 = run_island_evolution(
    initial_code=INITIAL_PROGRAM,
    evaluator=evaluate_sorting,
    mutator=mutate_random,
    num_islands=3,
    iterations=300,
    migration_interval=30,
)

# 可视化对比
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(single_best_history, label="单个 MAP-Elites 网格",
        color="#6b7280", linewidth=2)
ax.plot(hist3["best_score"], label="3 岛屿系统",
        color="#2563eb", linewidth=2)

ax.set_xlabel("迭代次数")
ax.set_ylabel("全局最佳分数")
ax.set_title("单网格 vs. 多岛屿：相同计算预算下的性能对比")
ax.legend()
plt.tight_layout()
plt.show()

single_final = single_best_history[-1] if single_best_history else 0
island_final = hist3["best_score"][-1] if hist3["best_score"] else 0
print(f"单网格最终最佳: {single_final:.4f}")
print(f"3 岛屿最终最佳: {island_final:.4f}")

total_diversity = sum(len(isl) for isl in system3.islands)
print(f"\n单网格保留程序数: {len(single_grid)}")
print(f"3 岛屿总保留程序数: {total_diversity}")

## OpenEvolve 中的岛屿配置

OpenEvolve 提供了多种预设的岛屿配置（`configs/island_examples.yaml`）：

| 配置名 | 岛屿数 | 迁移间隔 | 迁移比例 | 适用场景 |
|--------|--------|---------|---------|----------|
| `max_diversity` | 10 | 25 | 20% | 需要极大搜索空间的探索 |
| `focused_exploration` | 3 | 100 | 5% | 已知大方向，微调优化 |
| `balanced` | 5 | 50 | 10% | 通用默认配置 |
| `quick_exploration` | 3 | 20 | 15% | 快速原型，少量迭代 |
| `large_scale` | 15 | 75 | 8% | 大规模长时间运行 |

**经验法则：**
- 问题越复杂 → 更多岛屿 + 更低迁移比例
- 计算预算越少 → 更少岛屿 + 更频繁迁移
- 不确定时 → 用 `balanced` 配置

## 对应 OpenEvolve 源码

| 我们的实现 | OpenEvolve 源码 | 说明 |
|-----------|----------------|------|
| `IslandSystem.__init__` | `database.py:L122-176` | 岛屿初始化，OpenEvolve 用 `Set[str]` 存储程序ID |
| `IslandSystem.add()` | `database.py:L211-265` | 程序分配，OpenEvolve 还支持 `target_island` 参数 |
| `IslandSystem.sample()` | `database.py:L403-470` | 岛屿采样，OpenEvolve 有探索/利用/随机三种策略 |
| `IslandSystem.migrate()` | `database.py:L1775-1880` | 环形迁移，OpenEvolve 还有去重和防重复迁移 |
| `IslandSystem.should_migrate()` | `database.py:should_migrate()` | 基于代数的迁移触发逻辑完全一致 |
| `IslandSystem.next_island()` | `database.py:next_island()` | 轮转逻辑一致 |
| 迭代中的岛屿管理 | `process_parallel.py:L614-624` | OpenEvolve 用 `ProcessPoolExecutor` 并行各岛屿 |

## 保存到 `our-implementation/`

把岛屿系统保存为模块：

In [ ]:
MODULE_CODE = '''
"""our-implementation/islands.py — 多岛屿进化系统。

在第三章中构建。实现了岛屿模型 + MAP-Elites 的混合架构。
参考: Wright (1931) 种群遗传学, Tanese (1989) 并行遗传算法
"""
import random
import copy
from typing import Optional, List
from .core import Program
from .map_elites import MAPElitesGrid


class IslandSystem:
    """多岛屿进化系统，每个岛屿维护独立的 MAP-Elites 网格。"""

    def __init__(self, num_islands=5, feature_dims=None, n_bins=10,
                 migration_interval=50, migration_rate=0.1):
        if feature_dims is None:
            feature_dims = ["complexity", "score"]
        self.num_islands = num_islands
        self.migration_interval = migration_interval
        self.migration_rate = migration_rate
        self.islands = [MAPElitesGrid(feature_dims, n_bins) for _ in range(num_islands)]
        self.generations = [0] * num_islands
        self.last_migration_gen = 0
        self.current_island = 0
        self.global_best = None

    def add(self, program, island_id=None):
        idx = island_id if island_id is not None else self.current_island
        stored = self.islands[idx].add(program)
        program.metrics["island"] = idx
        fitness = program.metrics.get("score", 0.0)
        if self.global_best is None or fitness > self.global_best.metrics.get("score", 0.0):
            self.global_best = program
        return stored

    def sample(self, island_id=None):
        idx = island_id if island_id is not None else self.current_island
        if len(self.islands[idx]) > 0:
            return self.islands[idx].sample()
        for i in range(self.num_islands):
            if len(self.islands[i]) > 0:
                return self.islands[i].sample()
        raise ValueError("All islands empty")

    def next_island(self):
        self.current_island = (self.current_island + 1) % self.num_islands
        return self.current_island

    def increment_generation(self, island_id=None):
        idx = island_id if island_id is not None else self.current_island
        self.generations[idx] += 1

    def should_migrate(self):
        return (max(self.generations) - self.last_migration_gen) >= self.migration_interval

    def migrate(self):
        total = 0
        for k in range(self.num_islands):
            island = self.islands[k]
            if len(island) == 0:
                continue
            programs = sorted(island.grid.values(),
                              key=lambda p: p.metrics.get("score", 0.0), reverse=True)
            n_migrants = max(1, int(len(programs) * self.migration_rate))
            for prog in programs[:n_migrants]:
                if prog.metrics.get("migrant", False):
                    continue
                for neighbor in [(k+1) % self.num_islands, (k-1) % self.num_islands]:
                    migrant = Program(
                        code=prog.code,
                        metrics={**prog.metrics, "migrant": True, "source_island": k},
                        generation=prog.generation,
                    )
                    if self.islands[neighbor].add(migrant):
                        total += 1
        self.last_migration_gen = max(self.generations)
        return total

    def get_stats(self):
        stats = []
        for i, island in enumerate(self.islands):
            best_score = island.best.metrics.get("score", 0.0) if island.best else 0.0
            stats.append({"island": i, "population": len(island),
                          "coverage": f"{island.coverage:.0%}",
                          "best_score": f"{best_score:.4f}",
                          "generation": self.generations[i]})
        return stats
'''

with open("../our-implementation/islands.py", "w", encoding="utf-8") as f:
    f.write(MODULE_CODE)

print("已保存 our-implementation/islands.py")
print("  导出: IslandSystem")

## 本章小结

我们构建了：

- **`IslandSystem`** — 管理多个独立的 MAP-Elites 网格
- **轮转调度** — 轮流在各岛屿上进化
- **环形迁移** — 周期性地将精英程序复制到相邻岛屿

### 核心要点

1. **隔离保护创新** — 新颖但不成熟的方法不会被成熟方法淘汰
2. **迁移传播优秀基因** — 好的解决方案最终会传遍所有岛屿
3. **环形拓扑** 比全连接更好——给每个岛屿独立探索的时间
4. **防重复迁移** — 避免同一个程序被反复复制
5. **基于代数触发迁移** — 而非迭代次数，因为不同岛屿进化速度可能不同

### 还缺什么？

到目前为止，我们的「变异器」还是随机修改代码——这就像猴子在键盘上乱打。真正的 OpenEvolve 使用 **LLM 作为智能变异算子**，它能理解代码语义，生成有意义的修改。

**下一章：** [第四章 — LLM 作为变异算子](./04-llm-mutator.ipynb)  
我们将集成 LLM，让变异从「随机乱改」变成「理解后改进」。